In [20]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import scale
from statsmodels.nonparametric.smoothers_lowess import lowess
import os

In [19]:
manifest = pd.read_csv('/data/projects/liquid_biopsy/Projects/cfDNA/cfDNA/data/manifest/Cristiano_manifest.csv')
gc_content = pd.read_csv('/data/projects/liquid_biopsy/Projects/cfDNA/cfDNA/data/processing/gc_content_per_region.csv')
sample_ids = manifest['seqrun_id'].apply(lambda x: f'EE{x}').values

print(f"Total samples: {len(sample_ids)}")
print(f"First few samples: {sample_ids[:5]}")


Total samples: 459
First few samples: <StringArray>
['EE87786', 'EE87787', 'EE87788', 'EE87789', 'EE87790']
Length: 5, dtype: str


In [14]:

# Function to load from individual files
def load_from_individuals(feature_name, sample_ids):
    """Load feature matrix from individual sample files"""
    features_path = '/data/projects/liquid_biopsy/Projects/cfDNA/cfDNA/data/cristiano_features/'
    rows = []
    
    for i, sample_id in enumerate(sample_ids):
        if i % 50 == 0:
            print(f"Loading {i+1}/{len(sample_ids)}...")
        
        file_path = f"{features_path}{feature_name}/{sample_id}_{feature_name}.csv"
        df = pd.read_csv(file_path, index_col=0)
        
        # Flatten to 1D array
        values = df.values.flatten()
        rows.append(values)
    
    X = np.array(rows)
    return X


In [15]:

# Load PFE
print("\nLoading PFE features...")
sample_ids_test = sample_ids[:50]
X_pfe = load_from_individuals('pfe', sample_ids_test)

# Print info
print(f"\nMatrix shape: {X_pfe.shape}")
print(f"Data type: {X_pfe.dtype}")
print(f"\nFirst sample (first 10 values):")
print(X_pfe[0, :10])
print(f"\nSample statistics:")
print(f"Mean: {X_pfe.mean():.4f}")
print(f"Std: {X_pfe.std():.4f}")
print(f"Min: {X_pfe.min():.4f}")
print(f"Max: {X_pfe.max():.4f}")

# Check for NaN or Inf
print(f"\nNaN values: {np.isnan(X_pfe).sum()}")
print(f"Inf values: {np.isinf(X_pfe).sum()}")

print("\nDone")


Loading PFE features...
Loading 1/50...

Matrix shape: (50, 561414)
Data type: float64

First sample (first 10 values):
[2.52164064 1.5849625  2.23592635 0.         0.         1.93626003
 1.91829583 1.5        1.         1.        ]

Sample statistics:
Mean: 1.0692
Std: 0.7645
Min: -0.0000
Max: 3.8786

NaN values: 0
Inf values: 0

Done


In [17]:
df_pfe = pd.DataFrame(X_pfe)
df_pfe.head()

,0,1,2,3,4,5,6,7,8,9,...,561404,561405,561406,561407,561408,561409,561410,561411,561412,561413
0,2.521641,1.584963,2.235926,0.000000,0.000000,1.936260,1.918296,1.500000,1.000000,1.000000,...,1.000000,-0.0,-0.000000,2.0,1.000000,-0.000000,1.000000,0.0,0.0,-0.0
1,2.405639,2.625815,1.000000,-0.000000,1.584963,2.500000,2.663533,2.321928,1.584963,1.521928,...,1.500000,0.0,0.918296,1.0,1.584963,1.000000,1.584963,-0.0,1.0,1.0
2,1.000000,-0.000000,-0.000000,-0.000000,1.000000,2.549523,1.905639,1.918296,1.921928,2.128085,...,-0.000000,-0.0,-0.000000,-0.0,-0.000000,1.000000,-0.000000,-0.0,0.0,-0.0
3,1.584963,-0.000000,0.000000,1.584963,2.000000,1.657743,1.721928,1.500000,1.842371,2.807355,...,-0.000000,0.0,0.000000,0.0,1.500000,1.000000,-0.000000,0.0,-0.0,-0.0
4,2.500000,1.405639,0.918296,1.918296,0.918296,2.521641,2.565448,0.000000,1.584963,2.446439,...,1.842371,-0.0,-0.000000,1.0,1.000000,1.521928,0.000000,0.0,0.0,1.5


In [21]:
# normalize
X_pfe_scaled = scale(X_pfe)
print(f"\nScaled matrix shape: {X_pfe_scaled.shape}")
print(f"Scaled data type: {X_pfe_scaled.dtype}")
print(f"\nFirst sample (first 10 values) after scaling:")
print(X_pfe_scaled[0, :10])


Scaled matrix shape: (50, 561414)
Scaled data type: float64

First sample (first 10 values) after scaling:
[ 0.66287589 -0.15202562  1.39256574 -1.62474865 -1.28053829 -0.38885517
  0.01859735  0.10233988 -0.54455123 -0.65367033]


In [23]:
# gc correction using lowess span 0.75
gc_values = gc_content['gc_content'].values
gc_corrected = lowess(X_pfe_scaled.flatten(), gc_values, frac=0.75, return_sorted=False)
X_pfe_gc_corrected = X_pfe_scaled.flatten() - gc_corrected
X_pfe_gc_corrected = X_pfe_gc_corrected.reshape(X_pfe_scaled.shape)
print(f"\nGC-corrected matrix shape: {X_pfe_gc_corrected.shape}")
print(f"GC-corrected data type: {X_pfe_gc_corrected.dtype}")
print(f"\nFirst sample (first 10 values) after GC correction:")
print(X_pfe_gc_corrected[0, :10])


ValueError: exog and endog must have same length